# E3.6 · Saying no, and saying yes with conditions

**Function E — Governance, Risk, Compliance & the CISO Office → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

---

**Risk.** Conditional approval that is aspirational rather than enforceable.

**Control.** Autonomy promotion as an earned event with named evidence.

**This lab.** Make a conditional approval enforceable rather than aspirational.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E3.6"))

Saying no is cheap and rarely correct. Saying yes with conditions is the actual job, and the conditions have to be testable.

In [ ]:
from cybercommons import planes, grc
W = planes.Tool

ask = planes.Manifest("customer-refund-agent", [
    W("read_file"),
    W("issue_refund", writes=True, scope="tenant", reversible=False)], rung="L2.5")
asset = grc.AIAsset("customer-refund-agent", "agent", owner="payments-eng",
                    autonomy="L2.5", data=("customer", "regulated"))

print("request:", ask.agent, "at", ask.rung)
print("tier:", grc.risk_tier(asset)["tier"],
      " blast:", ask.blast_radius()["total"])
for p in ask.rung_check():
    print("  ⚠", p)

print("\nyes, with conditions:")
CONDITIONS = [
 ("refund cap per action, enforced in the tool", "bounds the irreversible step"),
 ("approval gate above the cap",                 "SB-2, testable in the policy file"),
 ("act chain on every refund",                   "AC-1/EV-1, testable in the logs"),
 ("tested stop, measured in seconds",            "ST-1, testable at a game day"),
 ("re-tier if the tool list changes",            "A1.1 manifest diff in CI"),
]
for cond, why in CONDITIONS:
    print(f"   · {cond:44s} {why}")

Now show the condition working, because a condition you cannot demonstrate is a condition nobody will meet.

In [ ]:
gated = planes.Manifest("customer-refund-agent", ask.tools,
                        approval_required={"issue_refund"}, rung="L2.5")
print("blast radius with the gate:", gated.blast_radius()["total"],
      "  issues:", gated.rung_check() or "none")

### Expect

The request tiers critical with a flagged irreversible ungated tool. Five testable conditions print, and applying the approval gate drops the blast radius to 0 with no remaining issues.

### Your turn

Take a request you refused in the last year and write the five conditions that would have made it a yes. Send them to the team that asked.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E3.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*